# Deep Crypto Policy GPU Training

Runtime: GPU. This notebook runs the repo's two-stage transformer policy: pretrain forecast/vol/regime heads, then fine-tune oracle action/ranking heads.

Run order: setup, CUDA check, Faster Debug Run, then Full GPU Run.

In [11]:
REPO = '/content/vapar'
BRANCH = 'codex/deep-crypto-colab-training'

%cd /content
!rm -rf "$REPO"
!git clone --depth 1 --branch "$BRANCH" https://github.com/comprono/vapar.git "$REPO"
%cd "$REPO"
!pip -q install -r backend/requirements.txt

/content
Cloning into '/content/vapar'...
remote: Enumerating objects: 368, done.
remote: Counting objects: 100% (368/368), done.
remote: Compressing objects: 100% (334/334), done.
remote: Total 368 (delta 25), reused 362 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (368/368), 7.09 MiB | 16.88 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/content/vapar


In [13]:
import torch, os
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

torch 2.10.0+cu128
cuda True
Tesla T4


## Faster Debug Run
Run this first. It proves the Colab runtime, data fetch, labels, pretraining, fine-tuning, ranking loss, checkpoints, and reports are all working before spending hours on the larger run.

In [14]:
!python -u tests/run_deep_crypto_policy_walkforward.py \
  --years 3 \
  --window-size 96 \
  --train-lookback-days 900 \
  --calibration-lookback-days 240 \
  --min-train-samples 1024 \
  --max-train-samples 12000 \
  --max-months 12 \
  --pretrain-epochs 8 \
  --epochs 12 \
  --batch-size 256 \
  --d-model 96 \
  --n-heads 4 \
  --n-layers 3 \
  --ranking-weight 0.8 \
  --initial-capital 10

[DataLoader] Fetching 10 symbols with 10 threads...
[DataLoader] Fetching BTC-USD from Yahoo Finance
[DataLoader] Fetching ETH-USD from Yahoo Finance
[DataLoader] Fetching BNB-USD from Yahoo Finance
[DataLoader] Fetching SOL-USD from Yahoo Finance
[DataLoader] Fetching XRP-USD from Yahoo Finance
[DataLoader] Fetching DOGE-USD from Yahoo Finance[DataLoader] Fetching ADA-USD from Yahoo Finance

[DataLoader] Fetching TRX-USD from Yahoo Finance
[DataLoader] Fetching AVAX-USD from Yahoo Finance
[DataLoader] Fetching DOT-USD from Yahoo Finance
[DataLoader] Cached 1096 bars for DOT-USD
[DataLoader] Cached 1096 bars for ETH-USD
[DataLoader] Cached 1096 bars for BTC-USD
[DataLoader] Cached 1096 bars for DOGE-USD
[DataLoader] Cached 1096 bars for TRX-USD
[DataLoader] Cached 1096 bars for ADA-USD
[DataLoader] Cached 1096 bars for BNB-USD
[DataLoader] Cached 1096 bars for XRP-USD
[DataLoader] Cached 1096 bars for SOL-USD
[DataLoader] Cached 1096 bars for AVAX-USD
[DataLoader] Completed in 0.73s. L

## Full GPU Run
Run this only after Faster Debug completes and writes a report. This is intentionally much larger than the local CPU launcher.

In [ ]:
!python -u tests/run_deep_crypto_policy_walkforward.py \
  --years 8 \
  --window-size 128 \
  --train-lookback-days 1460 \
  --calibration-lookback-days 365 \
  --min-train-samples 2048 \
  --max-train-samples 100000 \
  --max-months 36 \
  --pretrain-epochs 40 \
  --epochs 55 \
  --batch-size 512 \
  --d-model 256 \
  --n-heads 16 \
  --n-layers 8 \
  --dropout 0.10 \
  --lr 0.0003 \
  --ranking-weight 1.0 \
  --initial-capital 10

[DataLoader] Fetching 10 symbols with 10 threads...
[DataLoader] Loading BTC-USD from cache
[DataLoader] Loading ETH-USD from cache
[DataLoader] Loading XRP-USD from cache[DataLoader] Loading BNB-USD from cache
[DataLoader] Loading SOL-USD from cache[DataLoader] Loading ADA-USD from cache


[DataLoader] Loading TRX-USD from cache
[DataLoader] Loading AVAX-USD from cache
[DataLoader] Loading DOGE-USD from cache
[DataLoader] Loading DOT-USD from cache
[DataLoader] Completed in 0.33s. Loaded 10/10 symbols.
[deep-policy] month 1/36 2023-05 train=2019-05-02..2023-05-01
[deep-policy] 2023-05 calibration pretrain 1/40 loss=2.619534 samples=8882
[deep-policy] 2023-05 calibration pretrain 2/40 loss=0.934280 samples=8882
[deep-policy] 2023-05 calibration pretrain 3/40 loss=0.926439 samples=8882
[deep-policy] 2023-05 calibration pretrain 4/40 loss=0.933077 samples=8882
[deep-policy] 2023-05 calibration pretrain 5/40 loss=0.922086 samples=8882
[deep-policy] 2023-05 calibration pretrain 6/40 loss=0